# Lab 1: Basic Translation Agent - Understanding the Problem (15 minutes)

In this lab, you will create a basic translation agent and discover why simple one-shot translation isn't enough for professional-quality output.

## Learning Objectives

**You'll Learn These Core Patterns:**
- ✅ Create a Strands agent with Bedrock: `Agent(model=BedrockModel(...))`
- ✅ Perform basic translation tasks
- ✅ Identify quality issues in machine translation
- ✅ Understand why iteration is needed


> ⚠️ **EDUCATIONAL PURPOSE ONLY**: This workshop demonstrates intelligent translation patterns using AWS documentation. The techniques shown are for learning agentic AI patterns, not for production translation services.

## Step 1: Install and Import Dependencies

**Core Pattern**: Setup Strands SDK and AWS Bedrock

In [1]:
# Install required packages
!pip install -q -r requirements.txt

In [2]:
from strands import Agent
from strands.models import BedrockModel
import sys
sys.path.append('lab_helpers')
from utils import print_section_header, print_success, print_info

print_success("Strands Agents SDK imported successfully!")
print_info("AWS Workshop: Ready to build your translation agent!")

✅ Strands Agents SDK imported successfully!
ℹ️  AWS Workshop: Ready to build your translation agent!


## Step 2: Load Sample AWS Documentation

We'll use real AWS Lambda documentation as our translation source.

In [3]:
# Load sample AWS documentation
with open('sample_data/aws_lambda_intro.txt', 'r', encoding='utf-8') as f:
    aws_doc_text = f.read()

print_section_header("Source Document (English)")
print(aws_doc_text)
print(f"\n📊 Document length: {len(aws_doc_text)} characters")


  Source Document (English)

AWS Lambda Overview

AWS Lambda is a serverless compute service that lets you run code without provisioning or managing servers. Lambda runs your code on high-availability compute infrastructure and performs all the administration of the compute resources, including server and operating system maintenance, capacity provisioning and automatic scaling, and logging.

With Lambda, you can run code for virtually any type of application or backend service. You organize your code into Lambda functions. The Lambda service runs your function only when needed and scales automatically. You only pay for the compute time that you consume—there's no charge when your code isn't running.

Key Features:
- Automatic scaling based on incoming requests
- Built-in fault tolerance and high availability
- Pay-per-use pricing model with no charges for idle time
- Integration with other AWS services through event sources
- Support for multiple programming languages including Pytho

## Step 3: Configure AWS Bedrock Model

**Core Strands Pattern**: `BedrockModel(model_id=...)`

In [4]:
# Configure Bedrock model
model = BedrockModel(
    model_id="us.anthropic.claude-3-7-sonnet-20250219-v1:0"
)

print_success("AWS Bedrock model configured!")
print_info("Using Claude 3.7 Sonnet for translation")

✅ AWS Bedrock model configured!
ℹ️  Using Claude 3.7 Sonnet for translation


## Step 4: Create Basic Translation Agent

**Core Strands Pattern**: `Agent(model=model, system_prompt="...")`

In [5]:
# Create a basic translation agent
basic_translator = Agent(
    model=model,
    system_prompt="""
    You are a translation assistant. Translate the complete text from English to Russian.
    Translate all sentences and paragraphs provided.
    Return only the translated text without any explanations or commentary.
    """
)

print_success("Basic translation agent created!")

✅ Basic translation agent created!


## Step 5: Perform One-Shot Translation

Let's see what a basic translation looks like.

In [6]:
print_section_header("One-Shot Translation (No Refinement)")

# Translate the first paragraph only for demo
paragraphs = aws_doc_text.split('\n\n')
first_paragraph = paragraphs[1] if len(paragraphs) > 1 else paragraphs[0]  # Skip title, get first real paragraph

# Debug: show what we're translating
print(f"Source text to translate ({len(first_paragraph)} chars):")
print(f"{first_paragraph}\n")

translation_request = f"Translate this AWS documentation to Russian:\n\n{first_paragraph}"
result = basic_translator(translation_request)

print("\n📝 Translation Result:")
# Extract text from message
if isinstance(result.message, dict):
    content = result.message.get('content', [])
    if content and isinstance(content, list):
        print(content[0].get('text', result.message))
    else:
        print(result.message)
else:
    print(result.message)


  One-Shot Translation (No Refinement)

Source text to translate (344 chars):
AWS Lambda is a serverless compute service that lets you run code without provisioning or managing servers. Lambda runs your code on high-availability compute infrastructure and performs all the administration of the compute resources, including server and operating system maintenance, capacity provisioning and automatic scaling, and logging.

AWS Lambda - это бессерверная вычислительная служба, которая позволяет запускать код без выделения ресурсов или управления серверами. Lambda запускает ваш код на вычислительной инфраструктуре с высокой доступностью и выполняет все администрирование вычислительных ресурсов, включая обслуживание серверов и операционных систем, выделение мощностей и автоматическое масштабирование, а также ведение журналов.
📝 Translation Result:
AWS Lambda - это бессерверная вычислительная служба, которая позволяет запускать код без выделения ресурсов или управления серверами. Lambda запус

## Step 6: Identify Quality Issues

Let's manually review the translation and identify common problems.

In [7]:
print_section_header("Common Translation Quality Issues")

print("""
🔍 What to look for in the translation above:

1. ❌ Technical Terms:
   - Is "Lambda" translated literally? (It shouldn't be - it's a service name)
   - Is "serverless" translated correctly?
   - Are AWS service names preserved?

2. ❌ Fluency:
   - Does it sound natural to native Russian speakers?
   - Are sentence structures awkward?
   - Is the tone appropriate for technical documentation?

3. ❌ Consistency:
   - Are technical terms translated consistently?
   - Is terminology aligned with AWS's official Russian docs?

4. ❌ Accuracy:
   - Does it convey the exact same meaning?
   - Are any concepts lost or misrepresented?
""")

print("\n💡 Key Insight: One-shot translation often produces technically correct")
print("   but unnatural or inconsistent output. Professional translation requires")
print("   iteration and refinement.")


  Common Translation Quality Issues


🔍 What to look for in the translation above:

1. ❌ Technical Terms:
   - Is "Lambda" translated literally? (It shouldn't be - it's a service name)
   - Is "serverless" translated correctly?
   - Are AWS service names preserved?

2. ❌ Fluency:
   - Does it sound natural to native Russian speakers?
   - Are sentence structures awkward?
   - Is the tone appropriate for technical documentation?

3. ❌ Consistency:
   - Are technical terms translated consistently?
   - Is terminology aligned with AWS's official Russian docs?

4. ❌ Accuracy:
   - Does it convey the exact same meaning?
   - Are any concepts lost or misrepresented?


💡 Key Insight: One-shot translation often produces technically correct
   but unnatural or inconsistent output. Professional translation requires
   iteration and refinement.


## Step 7: Test with Different Prompts

Try improving the translation with better prompts.

In [8]:
# Create an improved translator with more detailed instructions
improved_translator = Agent(
    model=model,
    system_prompt="""
    You are a professional technical translator specializing in AWS documentation.
    
    Translation Rules:
    - Translate from English to Russian
    - Keep AWS service names in English (Lambda, S3, DynamoDB, etc.)
    - Use natural, fluent Russian that sounds native
    - Maintain technical accuracy
    - Use appropriate formal tone for technical documentation
    
    Provide only the translation.
    """
)

print_section_header("Improved Translation (Better Prompt)")

result2 = improved_translator(translation_request)
print("\n📝 Improved Translation:")
print(result2.message)


  Improved Translation (Better Prompt)

AWS Lambda — это бессерверный вычислительный сервис, который позволяет запускать код без выделения ресурсов или управления серверами. Lambda выполняет ваш код на вычислительной инфраструктуре с высокой доступностью и осуществляет все административные задачи, связанные с вычислительными ресурсами, включая обслуживание серверов и операционных систем, выделение мощностей и автоматическое масштабирование, а также ведение журналов.
📝 Improved Translation:
{'role': 'assistant', 'content': [{'text': 'AWS Lambda — это бессерверный вычислительный сервис, который позволяет запускать код без выделения ресурсов или управления серверами. Lambda выполняет ваш код на вычислительной инфраструктуре с высокой доступностью и осуществляет все административные задачи, связанные с вычислительными ресурсами, включая обслуживание серверов и операционных систем, выделение мощностей и автоматическое масштабирование, а также ведение журналов.'}]}


## Step 8: Compare Translations

Even with better prompts, one-shot translation has limitations.

In [9]:
print_section_header("Comparison: Basic vs Improved")

print("🤔 Questions to consider:")
print("")
print("1. Is the improved version better? In what ways?")
print("2. Are there still issues with terminology consistency?")
print("3. Does it sound natural to a native Russian speaker?")
print("4. How would you know if specific technical terms are correct?")
print("")
print("💡 The Problem: Even with perfect prompts, the agent can't:")
print("   - Verify its own translation quality")
print("   - Check terminology against authoritative sources")
print("   - Refine awkward phrasing iteratively")
print("   - Learn from its mistakes in real-time")


  Comparison: Basic vs Improved

🤔 Questions to consider:

1. Is the improved version better? In what ways?
2. Are there still issues with terminology consistency?
3. Does it sound natural to a native Russian speaker?
4. How would you know if specific technical terms are correct?

💡 The Problem: Even with perfect prompts, the agent can't:
   - Verify its own translation quality
   - Check terminology against authoritative sources
   - Refine awkward phrasing iteratively
   - Learn from its mistakes in real-time


## Lab 1 Complete - Key Takeaways

### 🎉 What You've Learned:

✅ **Core Strands Patterns:**
- `Agent(model=BedrockModel(...))` - Basic agent creation
- System prompts for specialization
- One-shot task execution

✅ **Translation Challenges Identified:**
- Technical terminology inconsistency
- Lack of fluency in target language
- No self-evaluation mechanism
- No access to authoritative terminology sources

✅ **Why Iteration Matters:**
- Professional translation requires multiple passes
- Quality improves with self-evaluation and refinement
- Agents need tools to verify and improve their work

**🎯 The Gap:** Current agent does translation but can't evaluate or improve its own output.

### 🚀 Next: Lab 2 - Self-Evaluation and Iteration

In Lab 2, you'll build an agent that can:
- Evaluate its own translation quality
- Identify specific issues
- Iteratively refine until quality threshold is met
- Show its reasoning process
